### Genollama: Model fine-tuning

Set up environment (including `assume --env` credentials if required)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

In [ ]:
from finetune import HuggingFaceLoRATrainer
from genoschema.prompt_builder import PromptBuilder
from genoschema.schema import GenomicTestReport

#### Start fine-tuning run

In [ ]:
model_name = "genoqwen"
trainer = HuggingFaceLoRATrainer(
    schema=GenomicTestReport,
    prompt_builder=PromptBuilder(),
    training_batch_names=["20260125-191050_genollama-batch"],
    hyperparameters={
        "base_model": "Qwen/Qwen3-4B-Instruct-2507",
        "num_epochs": 3,
        "learning_rate": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_target_modules": "q_proj,k_proj,v_proj,o_proj",
        "per_device_train_batch_size": 4,
        "max_seq_length": 2048,
    },
    aws_config={
        "bucket": os.environ["BUCKET"],
        "region": os.environ["AWS_REGION"],
        "role": os.environ["SAGEMAKER_EXECUTION_ROLE"],
    },
    model_name=model_name,
    description=f"{model_name}-lora",
    instance_type=os.environ["TRAINING_INSTANCE_TYPE"],
)

In [ ]:
trainer.run()

#### Post-process and upload model

In [ ]:
from huggingface_hub import logging
logging.set_verbosity_info()

In [ ]:
trainer.post_process(
    trainer.create_model_card(1, 0, 0), 
    "jobs/train/20260310-161448-genoqwen-lora/output", 
    "mesa-20260310-161448-genoqwen-lora-2026-03-10-16-14-55-305"
)